<a href="https://colab.research.google.com/github/r-yv/PromptEng/blob/main/Prompt_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║          Prompt Chain — Interactive Customer Support AI                      ║
║          Tool: Python (Standard Library) — Google Colab Compatible           ║
╚══════════════════════════════════════════════════════════════════════════════╝

Prompt 1: Create a simple e-commerce support system for customers to interact
Prompt 2: Now enable options that people can choose from, to make it feel like a real system
Prompt 3: Finalize the output by having the user be redirected and produce an output of the scenario provided

Iteration 1: Basic functionality support system, doesn't have options supported
Iteration 2: Now includes options to choose from, but redirectory isn't polished
Iteration 3: Functionality support of a basic customer support system, streamlined with a support agent
"""

import re
import sys
import textwrap
from dataclasses import dataclass, field
from typing import Optional, List


# ══════════════════════════════════════════════════════════════════════════════
# MOCK DATA
# ══════════════════════════════════════════════════════════════════════════════

ORDER_DB = {
    "ORD-1001": {
        "status": "shipped",    "item": "Wireless Headphones",
        "tracking": "UPS-4839201XZ", "carrier": "UPS",
        "shipped_date": "2026-02-24", "est_delivery": "2026-03-01",
    },
    "ORD-1002": {
        "status": "processing", "item": "Mechanical Keyboard",
        "tracking": None,       "carrier": None,
        "shipped_date": None,   "est_delivery": "2026-03-04",
    },
    "ORD-1003": {
        "status": "delivered",  "item": "4K Webcam",
        "tracking": "FDX-0029", "carrier": "FedEx",
        "shipped_date": "2026-02-18", "est_delivery": "2026-02-21",
    },
    "ORD-1004": {
        "status": "cancelled",  "item": "USB-C Hub",
        "tracking": None,       "carrier": None,
        "shipped_date": None,   "est_delivery": None,
    },
}

POLICIES = {
    "returns":      "30-day returns for unused items in original packaging. Free return shipping for defective goods.",
    "refunds":      "Refunds issued within 3-5 business days to the original payment method.",
    "shipping":     "Free standard shipping (5-7 days) on orders over $50. Express 2-day: $9.99.",
    "cancellation": "Orders cancellable within 1 hour of placement only.",
    "damaged":      "Damaged item? Contact us within 48 hours with a photo for a free replacement or full refund.",
    "warranty":     "All electronics include a 1-year manufacturer warranty covering defects.",
}


# ══════════════════════════════════════════════════════════════════════════════
# MENU STRUCTURE
# Each top-level category maps to a list of sub-issues the customer can pick.
# The chosen sub-issue is injected into the chain as the "raw_message" so the
# chain logic runs identically whether input came from the menu or free text.
# ══════════════════════════════════════════════════════════════════════════════

MENU = {
    "1": {
        "label": "Order Status & Tracking",
        "category": "order_status",
        "sub": {
            "1": "Where is my order? It hasn't arrived yet.",
            "2": "My order shows as delivered but I didn't receive it.",
            "3": "I need my tracking number.",
            "4": "My order is still in processing — when will it ship?",
        },
    },
    "2": {
        "label": "Returns & Refunds",
        "category": "returns_refunds",
        "sub": {
            "1": "I want to return an item and get a refund.",
            "2": "I returned my item — where is my refund?",
            "3": "I received the wrong item and want to exchange it.",
            "4": "What is your return policy?",
        },
    },
    "3": {
        "label": "Damaged or Defective Item",
        "category": "product_issue",
        "sub": {
            "1": "My item arrived damaged or broken.",
            "2": "My item stopped working after a short time.",
            "3": "I received the wrong item entirely.",
            "4": "My item is missing parts or accessories.",
        },
    },
    "4": {
        "label": "Order Cancellation",
        "category": "cancellation",
        "sub": {
            "1": "I want to cancel my order — I just placed it.",
            "2": "I want to cancel but my order is already processing.",
            "3": "I ordered the wrong item and need to cancel.",
        },
    },
    "5": {
        "label": "Shipping & Delivery Info",
        "category": "shipping_info",
        "sub": {
            "1": "How much does shipping cost?",
            "2": "Do you offer free shipping?",
            "3": "How long does standard delivery take?",
            "4": "Do you offer express or next-day delivery?",
        },
    },
    "6": {
        "label": "Billing & Payments",
        "category": "payment",
        "sub": {
            "1": "I was charged incorrectly for my order.",
            "2": "My payment was declined.",
            "3": "I need a copy of my invoice.",
            "4": "I want to update my payment method.",
        },
    },
    "7": {
        "label": "Something Else",
        "category": "general",
        "sub": {
            "1": "I have a question not listed above.",
        },
    },
}

# Sub-issues that always need an order ID to proceed
NEEDS_ORDER_ID = {
    "order_status", "returns_refunds", "product_issue", "cancellation",
}


# ══════════════════════════════════════════════════════════════════════════════
# SHARED STATE
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Context:
    raw_message:         str

    # Step 1 outputs
    category:            Optional[str] = None
    sentiment:           Optional[str] = None
    priority:            Optional[str] = None
    order_id:            Optional[str] = None
    customer_name:       Optional[str] = None

    # Step 2 outputs
    order_record:        Optional[dict] = None
    policy_text:         Optional[str]  = None
    missing_info:        Optional[str]  = None

    # Step 3 outputs
    resolution:          Optional[str] = None
    reply:               Optional[str] = None

    # Step 4 outputs
    qa_passed:           bool          = False
    qa_notes:            List[str]     = field(default_factory=list)
    escalate:            bool          = False
    final_reply:         Optional[str] = None


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — CLASSIFY
#
# PROMPT:
#   "Classify the customer's issue. Return: category, sentiment, priority,
#    order_id (if mentioned), customer_name (if introduced).
#    Constraints: angry → urgent priority. Product issues → high priority always."
#
# INPUT  : ctx.raw_message
# OUTPUT : ctx.category, ctx.sentiment, ctx.priority, ctx.order_id, ctx.customer_name
# ══════════════════════════════════════════════════════════════════════════════

def step1_classify(ctx: Context) -> Context:
    msg = ctx.raw_message.lower()

    # Order ID extraction
    m = re.search(r'\bord-?\d{3,6}\b', msg, re.IGNORECASE)
    if m:
        digits = re.sub(r'[^0-9]', '', m.group(0))
        ctx.order_id = f"ORD-{digits}"

    # Customer name extraction
    nm = re.search(r'\bmy name is ([A-Z][a-z]+)\b', ctx.raw_message, re.IGNORECASE)
    if nm:
        ctx.customer_name = nm.group(1).title()

    # Category (most specific rule wins — already set by menu, but re-confirmed here)
    if ctx.category is None:
        if any(w in msg for w in ["cancel"]):
            ctx.category = "cancellation"
        elif any(w in msg for w in ["broken", "damaged", "defective", "not working", "wrong item"]):
            ctx.category = "product_issue"
        elif any(w in msg for w in ["return", "refund", "send back", "money back", "exchange"]):
            ctx.category = "returns_refunds"
        elif any(w in msg for w in ["free shipping", "shipping cost", "express", "how long"]):
            ctx.category = "shipping_info"
        elif any(w in msg for w in ["where is", "tracking", "not arrived", "still waiting"]):
            ctx.category = "order_status"
        elif any(w in msg for w in ["charge", "billed", "payment", "invoice"]):
            ctx.category = "payment"
        else:
            ctx.category = "general"

    # Sentiment — inferred from any free-text the user added
    if any(w in msg for w in ["unacceptable", "furious", "ridiculous", "worst",
                               "outraged", "scam", "terrible", "!!!"]):
        ctx.sentiment = "angry"
    elif any(w in msg for w in ["frustrated", "annoyed", "disappointed",
                                  "waiting weeks", "still waiting", "again"]):
        ctx.sentiment = "frustrated"
    elif any(w in msg for w in ["thank", "great", "happy", "love", "appreciate"]):
        ctx.sentiment = "positive"
    else:
        ctx.sentiment = "neutral"

    # Priority
    ctx.priority = {
        "angry":      "urgent",
        "frustrated": "high",
        "neutral":    "medium",
        "positive":   "low",
    }[ctx.sentiment]

    if ctx.category == "product_issue":
        ctx.priority = "high"

    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — GATHER INFO
#
# PROMPT:
#   "Look up the order record and relevant policy. If critical info is missing
#    (e.g., no order ID for an order-status issue), state exactly what to ask.
#    Constraints: never invent order data. missing_info = one clear question or null."
#
# INPUT  : ctx.category, ctx.order_id  (from Step 1)
# OUTPUT : ctx.order_record, ctx.policy_text, ctx.missing_info
# ══════════════════════════════════════════════════════════════════════════════

def step2_gather_info(ctx: Context) -> Context:
    # Order lookup
    if ctx.order_id:
        ctx.order_record = ORDER_DB.get(ctx.order_id)

    # Relevant policy
    policy_map = {
        "returns_refunds": "returns",
        "product_issue":   "damaged",
        "cancellation":    "cancellation",
        "shipping_info":   "shipping",
        "order_status":    "shipping",
        "payment":         "refunds",
    }
    ctx.policy_text = POLICIES.get(policy_map.get(ctx.category, ""), None)

    # What are we still missing?
    if ctx.category in NEEDS_ORDER_ID and not ctx.order_id:
        ctx.missing_info = (
            "Could you share your order ID? "
            "It looks like ORD-XXXX and is in your confirmation email."
        )
    elif ctx.order_id and ctx.order_record is None:
        ctx.missing_info = (
            f"I couldn't find order {ctx.order_id} in our system. "
            "Could you double-check the number?"
        )
    else:
        ctx.missing_info = None

    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — PROPOSE FIX
#
# PROMPT:
#   "Write a warm, concise reply as Alex from Customer Support. Use the
#    resolution plan and order data. Match tone to detected sentiment.
#    Constraints: do not invent tracking numbers or dates. No refund amount
#    guarantees. If missing_info is set, only ask that question."
#
# INPUT  : ctx.category, ctx.sentiment, ctx.order_record,
#          ctx.policy_text, ctx.missing_info  (from Steps 1–2)
# OUTPUT : ctx.resolution, ctx.reply
# ══════════════════════════════════════════════════════════════════════════════

def step3_propose_fix(ctx: Context) -> Context:
    name     = ctx.customer_name or "there"
    greeting = f"Hi {name},"
    empathy  = {
        "angry":      "I'm truly sorry about your experience and want to fix this immediately. ",
        "frustrated": "I'm sorry for the trouble — let me sort this out for you right away. ",
        "neutral":    "Thanks for reaching out! ",
        "positive":   "Happy to help! ",
    }[ctx.sentiment]

    # If we still need info, just ask for it
    if ctx.missing_info:
        ctx.resolution = "ask_for_missing_info"
        ctx.reply = (
            f"{greeting}\n\n"
            f"{empathy}"
            f"{ctx.missing_info}\n\n"
            "Warm regards,\nAlex | Customer Support"
        )
        return ctx

    o   = ctx.order_record or {}
    oid = ctx.order_id or "your order"

    if ctx.category == "order_status":
        status = o.get("status", "unknown")
        if status == "shipped":
            ctx.resolution = "provide_tracking"
            core = (
                f"Your order {oid} ({o.get('item','')}) shipped via {o.get('carrier','')} "
                f"on {o.get('shipped_date','')}. Tracking: {o.get('tracking','')}. "
                f"Expected delivery: {o.get('est_delivery','')}."
            )
        elif status == "processing":
            ctx.resolution = "update_processing"
            core = (
                f"Your order {oid} ({o.get('item','')}) is still being prepared. "
                f"It should dispatch by {o.get('est_delivery','')} — "
                "you'll get a tracking email the moment it ships."
            )
        elif status == "delivered":
            ctx.resolution = "confirm_delivered"
            core = (
                f"Our records show {oid} was delivered. "
                "Could you check with a neighbour or building reception? "
                "If it's still missing, reply here and I'll open an investigation."
            )
        else:
            ctx.resolution = "provide_status"
            core = f"Your order {oid} currently has status: {status}."

    elif ctx.category == "returns_refunds":
        ctx.resolution = "initiate_return"
        core = (
            f"You're within our 30-day return window for {oid}. "
            "I'll email you a prepaid return label within 24 hours. "
            "Once we receive the item, your refund processes in 3-5 business days."
        )

    elif ctx.category == "product_issue":
        ctx.resolution = "offer_replacement_or_refund"
        core = (
            f"I'm so sorry about your {oid}. "
            "We can send a free replacement straight away, "
            "or issue a full refund within 3-5 business days. Which would you prefer?"
        )

    elif ctx.category == "cancellation":
        if o.get("status") == "processing":
            ctx.resolution = "cannot_cancel"
            core = (
                f"Order {oid} has already entered fulfilment and can't be stopped now. "
                "When it arrives, simply refuse delivery — "
                "we'll refund you in full within 3-5 business days."
            )
        else:
            ctx.resolution = "cancel_order"
            core = (
                f"Done! Order {oid} has been cancelled and a full refund is on its way. "
                "Expect it on your statement within 3-5 business days."
            )

    elif ctx.category == "shipping_info":
        ctx.resolution = "provide_policy"
        core = ctx.policy_text or "Please visit our Help Centre for full shipping details."

    elif ctx.category == "payment":
        ctx.resolution = "escalate_billing"
        core = (
            "Billing matters need a specialist review. "
            "I've flagged your case as high priority — "
            "our billing team will contact you within 24 hours."
        )

    else:
        ctx.resolution = "general_info"
        core = (
            "I'd be happy to help with that! "
            "Could you share a little more detail so I can point you in the right direction?"
        )

    ctx.reply = (
        f"{greeting}\n\n"
        f"{empathy}{core}\n\n"
        "Warm regards,\nAlex | Customer Support"
    )
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — QA + ESCALATION CHECK
#
# PROMPT:
#   "Review the draft reply against 5 QA rules. Auto-fix missing sign-off.
#    Flag other failures without removing content. Hard rule: angry or urgent
#    cases always receive a human-escalation notice appended to the reply."
#
# INPUT  : ctx.reply, ctx.order_id, ctx.sentiment, ctx.priority  (Steps 1–3)
# OUTPUT : ctx.qa_passed, ctx.qa_notes, ctx.escalate, ctx.final_reply
# ══════════════════════════════════════════════════════════════════════════════

def step4_qa_and_escalate(ctx: Context) -> Context:
    draft = ctx.reply or ""
    notes = []

    if not re.search(r'\b(hi|hello)\b', draft, re.IGNORECASE):
        notes.append("QA-1 FAIL: Missing greeting.")

    if ctx.order_id and ctx.order_id not in draft:
        notes.append(f"QA-2 FAIL: Order ID {ctx.order_id} not in reply.")

    if not re.search(r'\b(will|can|email|send|refund|ship|contact|reply|check)\b',
                     draft, re.IGNORECASE):
        notes.append("QA-3 FAIL: No clear action verb or next step.")

    if "support" not in draft.lower():
        draft += "\n\nWarm regards,\nAlex | Customer Support"
        notes.append("QA-4 AUTO-FIXED: Added missing sign-off.")

    if len(draft.split()) < 20:
        notes.append("QA-5 FAIL: Reply is too short.")

    ctx.qa_passed = not any("FAIL" in n for n in notes)
    ctx.qa_notes  = notes

    # Hard escalation rule — angry or urgent always gets a human handoff
    if ctx.sentiment == "angry" or ctx.priority == "urgent":
        ctx.escalate = True
        draft += (
            "\n\n─────────────────────────────────────────\n"
            "Given the urgency of your situation, I've flagged your case "
            "for a senior support specialist who will contact you within 2 business hours."
        )

    ctx.final_reply = draft
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# CHAIN RUNNER
# ══════════════════════════════════════════════════════════════════════════════

STEPS = [
    ("1", "Classify",      step1_classify),
    ("2", "Gather Info",   step2_gather_info),
    ("3", "Propose Fix",   step3_propose_fix),
    ("4", "QA + Escalate", step4_qa_and_escalate),
]

def run_chain(message: str, category: str = None, verbose: bool = False) -> Context:
    ctx = Context(raw_message=message)
    if category:
        ctx.category = category   # pre-set from menu selection
    for _, _, fn in STEPS:
        ctx = fn(ctx)
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

W = 60   # display width

def _divider(char="─"):
    print(f"  {char * (W - 2)}")

def _print(text: str, indent: int = 2):
    for line in text.split("\n"):
        for wrapped in (textwrap.wrap(line, width=W - indent - 2) or [""]):
            print(" " * indent + wrapped)

def _input(prompt: str) -> str:
    """Wrapper around input() that handles EOF/interrupt gracefully."""
    try:
        return input(prompt).strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\n  Session ended. Goodbye!")
        sys.exit(0)

def _print_reply(ctx: Context):
    print()
    _divider("═")
    print("  SUPPORT REPLY")
    _divider("═")
    plain = (ctx.final_reply or "").replace("**", "").replace("`", "")
    _print(plain, indent=2)
    _divider("═")
    if ctx.escalate:
        print("  [Case flagged for senior specialist follow-up]")
    if ctx.qa_notes:
        print("  [QA notes (internal):]")
        for note in ctx.qa_notes:
            print(f"    • {note}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MENU NAVIGATION
# ══════════════════════════════════════════════════════════════════════════════

def show_main_menu():
    print()
    _divider("═")
    print("  CUSTOMER SUPPORT — MAIN MENU")
    _divider("═")
    print("  What can we help you with today?\n")
    for key, entry in MENU.items():
        print(f"    {key}.  {entry['label']}")
    print()
    print("    0.  Exit")
    _divider()


def show_sub_menu(entry: dict):
    print()
    _divider()
    print(f"  {entry['label'].upper()}")
    _divider()
    print("  Please select your specific issue:\n")
    for key, text in entry["sub"].items():
        # Wrap long sub-issue descriptions neatly
        lines = textwrap.wrap(text, width=W - 10)
        print(f"    {key}.  {lines[0]}")
        for extra in lines[1:]:
            print(f"         {extra}")
    print()
    print("    0.  Back to main menu")
    _divider()


def ask_order_id() -> Optional[str]:
    """Prompt the user for an order ID and return it normalised, or None."""
    print()
    print("  We'll need your order ID to look up your case.")
    print("  It looks like ORD-1001 and is in your confirmation email.")
    print("  (Available test IDs: ORD-1001, ORD-1002, ORD-1003, ORD-1004)")
    print("  Press Enter to skip if you don't have it handy.")
    print()
    raw = _input("  Order ID > ").strip()
    if not raw:
        return None
    m = re.search(r'\d{3,6}', raw)
    return f"ORD-{m.group(0)}" if m else None


def ask_extra_detail(sub_text: str) -> str:
    """Optionally collect more detail from the customer beyond the menu choice."""
    print()
    print("  Is there anything else you'd like to add? (Press Enter to skip)")
    extra = _input("  You > ").strip()
    if extra:
        return f"{sub_text} Additional detail: {extra}"
    return sub_text


def ask_name() -> Optional[str]:
    """Optionally collect the customer's first name for a personalised reply."""
    print()
    raw = _input("  Your first name (optional, press Enter to skip): ").strip()
    return raw.title() if raw else None


def run_session():
    """
    Full interactive support session:
      1. Greet the customer and ask their name.
      2. Show the main menu — let them pick a category.
      3. Show the sub-menu — let them pick the specific issue.
      4. Collect order ID if the category needs one.
      5. Offer a free-text field for extra detail.
      6. Run the 4-step chain and display the reply.
      7. Ask if they need help with anything else.
    """
    print()
    _divider("═")
    print("  Welcome to ShopEase Customer Support")
    print("  Powered by a 4-step AI prompt chain")
    _divider("═")

    # ── Optional name collection ──────────────────────────────────────────────
    name = ask_name()
    greeting = f"  Hi {name}! Let's get your issue sorted." if name else \
               "  Let's get your issue sorted."
    print(greeting)

    while True:
        # ── MAIN MENU ─────────────────────────────────────────────────────────
        show_main_menu()
        choice = _input("  Choose an option > ")

        if choice == "0":
            print("\n  Thank you for contacting ShopEase. Have a great day!")
            break

        if choice not in MENU:
            print("  Please enter a number from the menu.")
            continue

        entry    = MENU[choice]
        category = entry["category"]

        # ── SUB MENU ──────────────────────────────────────────────────────────
        while True:
            show_sub_menu(entry)
            sub_choice = _input("  Choose your issue > ")

            if sub_choice == "0":
                break   # back to main menu

            if sub_choice not in entry["sub"]:
                print("  Please enter a number from the list above.")
                continue

            sub_text = entry["sub"][sub_choice]

            # ── Order ID collection ───────────────────────────────────────────
            order_id = None
            if category in NEEDS_ORDER_ID:
                order_id = ask_order_id()

            # ── Optional extra detail ─────────────────────────────────────────
            message = ask_extra_detail(sub_text)

            # Inject name and order ID into the message string so the
            # classifier can pick them up naturally
            if name:
                message = f"My name is {name}. {message}"
            if order_id:
                message = f"{message} My order ID is {order_id}."

            # ── Run the 4-step chain ──────────────────────────────────────────
            print()
            _divider()
            print("  Processing your request through the support chain...")
            _divider()

            ctx = run_chain(message, category=category)
            _print_reply(ctx)

            # ── Follow-up ─────────────────────────────────────────────────────
            print()
            again = _input("  Is there anything else we can help with? (yes / no) > ").lower()
            if again in ("yes", "y"):
                break   # back to main menu
            else:
                print()
                _divider("═")
                print("  Thank you for contacting ShopEase!")
                print("  A transcript of this session has been saved.")
                if ctx.escalate:
                    print("  A specialist will follow up with you shortly.")
                _divider("═")
                return   # end session

        # If user chose '0' in sub-menu, loop back to main menu naturally


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    run_session()


  ══════════════════════════════════════════════════════════
  Welcome to ShopEase Customer Support
  Powered by a 4-step AI prompt chain
  ══════════════════════════════════════════════════════════

  Your first name (optional, press Enter to skip): John
  Hi John! Let's get your issue sorted.

  ══════════════════════════════════════════════════════════
  CUSTOMER SUPPORT — MAIN MENU
  ══════════════════════════════════════════════════════════
  What can we help you with today?

    1.  Order Status & Tracking
    2.  Returns & Refunds
    3.  Damaged or Defective Item
    4.  Order Cancellation
    5.  Shipping & Delivery Info
    6.  Billing & Payments
    7.  Something Else

    0.  Exit
  ──────────────────────────────────────────────────────────
  Choose an option > 3

  ──────────────────────────────────────────────────────────
  DAMAGED OR DEFECTIVE ITEM
  ──────────────────────────────────────────────────────────
  Please select your specific issue:

    1.  My item arrived 